# Manifold Studio: compose and inspect

Run after installing the repository with `pip install -e ".[viz]"`. This walkthrough uses synthetic numerical vectors; it makes no language-quality claim.

In [ ]:
import numpy as np
from manifold_studio import Sphere, Torus, Mobius, Plane, ManifoldProjector, Embedding, neighbor_preservation
rng = np.random.default_rng(42)
X = rng.normal(size=(120, 24))
texts = [f"Synthetic point {i}" for i in range(len(X))]

## Compose three geometries
A Cartesian product retains every factor. The combined space here is 6D intrinsically and uses 9 ambient coordinates. A 3D view is a projection.

In [ ]:
geometry = Sphere() * Torus() * Plane()
mapper = ManifoldProjector(geometry)
result = mapper.fit_transform(X, texts=texts)
print("intrinsic:", geometry.intrinsic_dim, "ambient:", geometry.ambient_dim)
result.plot_3d(target_index=0).show()

## Tangent directions and neighborhood preservation
Arrows are projected ambient displacements. Neighborhood overlap measures retained relations, not correctness of meaning.

In [ ]:
tangents = result.toward(0)
print(tangents.shape)
neighbor_preservation(X, result, k=5)

## Try a Möbius product
The current Möbius distance option is ambient chord distance, explicitly selected. The strip view respects its seam parametrization.

In [ ]:
other = ManifoldProjector(Torus() * Mobius()).fit_transform(X, texts=texts)
other.plot_3d(target_index=1).show()
neighbor_preservation(X, other, metric="ambient")

## Save and transform a future query
The trained/reference coordinate system is reused; a single-query batch does not refit it.

In [ ]:
from pathlib import Path
out = Path("outputs/notebook"); out.mkdir(parents=True, exist_ok=True)
result.save(out / "embeddings.npz")
mapper.save(out / "projector.npz")
restored = ManifoldProjector.load(out / "projector.npz")
np.testing.assert_allclose(restored.transform(X[:1]).coordinates, result.coordinates[:1])

## Use your own language embeddings
Replace X with an NPY matrix, or see `examples/text_demo.py` for the optional sentence-transformer adapter. These are sentence embeddings; contextual word extraction is a later milestone.